In [ ]:
import torch
from torch import nn

torch.cuda.empty_cache()

In [ ]:
from fusiontimeseries.experiments.config import FinetuningConfig

EVAL_CONTEXT_CUTOFF = 1
config = FinetuningConfig(
    eval_context_cutoff=EVAL_CONTEXT_CUTOFF,
    train_context_cutoffs=[1, 70, 139],
    subsampling=True,
    max_steps=400,
)

In [ ]:
from fusiontimeseries.experiments.config import get_output_dir


output_dir = get_output_dir("./outputs", base_folder=f"opc-test{EVAL_CONTEXT_CUTOFF}")

In [ ]:
from fusiontimeseries.experiments.dataset import FluxDataset

train_dataset = FluxDataset(namespaces=["gyroswin_train"], config=config)
val_dataset = FluxDataset(namespaces=["gyroswin_val"], config=config)
id_test_dataset = FluxDataset(namespaces=["gyroswin_id"], config=config)
ood_test_dataset = FluxDataset(namespaces=["gyroswin_ood"], config=config)
(
    train_dataset.samples[:3],
    val_dataset.samples[:3],
    id_test_dataset.samples[:3],
    ood_test_dataset.samples[:3],
)

In [ ]:
from fusiontimeseries.experiments.model import get_model
from fusiontimeseries.loralib.layers import BilinearLoRA

model: nn.Module = get_model(
    config=config, output_dir=output_dir, device="cuda", Adapter=BilinearLoRA
)

In [ ]:
model

In [ ]:
import json

from fusiontimeseries.experiments.trainer import TimesFMTrainer

trainer = TimesFMTrainer(
    model=model,  # type: ignore
    train_args=config.get_training_arguments(
        output_dir=output_dir, load_best_model_at_end=False
    ),
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    config=config,
)
with open(output_dir / "training_args.json", "w") as f:
    json.dump(trainer.args.to_dict(), f, indent=4)
config.save_config(output_dir / "fts_config.json")

In [ ]:
import json
from fusiontimeseries.loralib.utils import lora_state_dict

train_output = trainer.train()

with open(output_dir / "train_summary.json", "w") as f:
    json.dump(train_output._asdict(), f, indent=4)

lora_weights = lora_state_dict(model)
torch.save(lora_weights, output_dir / "lora_weights.pt")

In [ ]:
trained_model = trainer.model.eval()

In [ ]:
from fusiontimeseries.experiments.model import evaluate


id_results = evaluate(
    model=trained_model,
    config=config,
    data=id_test_dataset.flux_data,
    device="cuda",
)
with open(output_dir / "id_test_results.json", "w") as f:
    json.dump(id_results, f, indent=4)

ood_results = evaluate(
    model=trained_model,
    config=config,
    data=ood_test_dataset.flux_data,
    device="cuda",
)
with open(output_dir / "ood_test_results.json", "w") as f:
    json.dump(ood_results, f, indent=4)

val_results = evaluate(
    model=trained_model,
    config=config,
    data=val_dataset.flux_data,
    device="cuda",
)
with open(output_dir / "val_results.json", "w") as f:
    json.dump(val_results, f, indent=4)

train_results = evaluate(
    model=trained_model,
    config=config,
    data=train_dataset.flux_data,
    device="cuda",
)
with open(output_dir / "train_results.json", "w") as f:
    json.dump(train_results, f, indent=4)

In [ ]:
from fusiontimeseries.experiments.vizualization import plot_forecast

plot_forecast(
    results=id_results,
    simulations=[
        "3000",  # 8
        "3001",  # 115
    ],
    config=config,
    output_dir=output_dir,
    show_plots=True,
)
plot_forecast(
    results=val_results,
    simulations=[
        "2001",  # 100
        "2002",  # 200
    ],
    config=config,
    output_dir=output_dir,
    show_plots=True,
)
plot_forecast(
    results=ood_results,
    simulations=[
        "4001",  # 1
        "4003",  # 3
    ],
    config=config,
    output_dir=output_dir,
    show_plots=True,
)